In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.transforms import ScaledTranslation
import matplotlib.dates as mdates
import os
# os.chdir("/home/j/jackbarker/surface_predictor")
import pickle
import pandas as pd

import cartopy.crs as ccrs
import cartopy.feature as cfeature

def preprocess(ds):
    # Remove batch dimension if it exists
    if 'batch' in ds.dims:
        ds = ds.squeeze(dim="batch")
    if 'time' in ds.coords:
        ds = ds.rename({"time": "datetime"})
    return ds

def preprocess_era5_dataset(ds):
    ds = ds.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
    ds = ds.set_index({"datetime": "datetime"})
    ds = ds.reindex(lat=ds.lat[::-1])  # Reverse latitude order to match Graphcast data
    ds = ds.drop_vars(["number", "expver"])
    return ds

def average_area(start_date, end_date, tod, da):
    da = da.sel(lat=slice(7.5,13.5), lon=slice(35,41)).sel(datetime=slice(start_date, end_date))
    if tod!=-1:
        da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

def custom_average_area(start_date, end_date, tod, da, latslice, lonslice):
    da = da.sel(lat=latslice, lon=lonslice).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

def preprocess_ifs_data(ds):
    ds = ds.rename({
        'latitude': 'lat',
        'longitude': 'lon',
        'time': 'datetime',
        't2m': '2m_temperature',
    })
    ds = ds.reindex(lat = ds.lat[::-1])
    return ds


# Data loading

I combined all the t2m data from the 2014-16 GC runs into an error and forecast datafile, to make loading easier.

The `average_area` function creates the time series plots I've been making, by averaging all the points over a given area between a start and end date.

The below code implements the rolling average stuff, if you're interested (although I was no longer using this at the end). It works by doing a centred rolling average, and does not include the current point
```
def determine_bust_dates(timeseries, threshold=1.5, simple_threshold=False):
    if not simple_threshold:
        rolling_avg = timeseries.rolling(datetime=21, min_periods=1, center=True)

        rolling_avg = (rolling_avg.sum() - timeseries) / (rolling_avg.count()-1)

        delta = rolling_avg - timeseries

        std = timeseries.std()

        bust_dates = timeseries.datetime[delta > threshold * std]
    
    else:
        bust_dates = timeseries.datetime[timeseries < -threshold]

    return bust_dates
```

In [ ]:
all_error_data = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/gc_data/2mt_errors_2014_16.nc")
error_data_6am = all_error_data.where(all_error_data.datetime.dt.hour == 6, drop=True)
all_pred_data = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/gc_data/2mt_pred_2014_16.nc")
pred_data_6am = all_pred_data.where(all_pred_data.datetime.dt.hour == 6, drop=True)
era5_data = all_pred_data - all_error_data
era5_data_6am = era5_data.where(era5_data.datetime.dt.hour == 6, drop=True)

error_series_6am = average_area("2014-01-01", "2016-12-31", 6, error_data_6am["2m_temperature"])
gc_series_6am = average_area("2014-01-01", "2016-12-31", 6, pred_data_6am["2m_temperature"])
era5_series_6am = average_area("2014-01-01", "2016-12-31", 6, era5_data_6am["2m_temperature"])

In [ ]:
ifs_forecast = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data_lwda/da_forecasts2016_lwda.nc"))
# ifs_forecast_2 = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data/da_forecasts20160801-20161231.nc"))
ifs_analysis = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data_lwda/analysis2016_lwda.nc"))
# ifs_forecast = xr.concat([ifs_forecast_1, ifs_forecast_2], dim="datetime")
ifs_error = ifs_forecast - ifs_analysis
ifs_error_series_6am = average_area("2016-01-02", "2016-07-31", 6, ifs_error["2m_temperature"])
ifs_forecast_series_6am = average_area("2016-01-02", "2016-12-31", 6, ifs_forecast["2m_temperature"])
ifs_analysis_series_6am = average_area("2016-01-02", "2016-07-31", 6, ifs_analysis["2m_temperature"])


In [ ]:
bust_threshold = -1.5
bust_dates = error_series_6am.datetime[error_series_6am < bust_threshold]

# Main plots

## Plot of the time-series of stamp-sized maps

In [ ]:
# compute the anomalies
jan_2mt_anom = era5_data["2m_temperature"].sel(datetime = era5_data.datetime.dt.month==1)
jan_2mt_anom = jan_2mt_anom.sel(datetime = jan_2mt_anom.datetime.dt.hour == 6)
jan_2mt_anom = (jan_2mt_anom - jan_2mt_anom.mean(dim="datetime"))
jan_2mt_anom = jan_2mt_anom.sel(datetime = jan_2mt_anom.datetime.dt.hour == 6)

In [ ]:
# make the plot

def shift_cbar_ticklabels(cbar, dy_points=0):  
    # shifts top and bottom tick labels of a colourbar, so that neighbouring colourbars
    # do not clash
    # dy_points: +up, -down for vertical bar
    fig = cbar.ax.figure
    # Ensure renderer exists
    fig.canvas.draw_idle()
    dy = dy_points / 72.0  # convert points to inches
    for i, label in enumerate(cbar.ax.get_yticklabels()):
        if i==0:
            label.set_transform(label.get_transform() + ScaledTranslation(0, dy, fig.dpi_scale_trans))
        elif i==len(cbar.ax.get_yticklabels())-1:
            label.set_transform(label.get_transform() + ScaledTranslation(0, -dy, fig.dpi_scale_trans))

fig, axs = plt.subplots(
    3, 
    6, 
    figsize=(20, 8), 
    subplot_kw={"projection": ccrs.PlateCarree()},
    gridspec_kw={"wspace": -0.70, "hspace": 0.15})

start = 21

title_fontsize = 15
tick_fontsize = 15
cbar_title_fontsize = 15

# top row - ERA5 ---------------------------------------------------------------------


for i, ax in enumerate(axs[0]):
    data = era5_data["2m_temperature"].sel(datetime=f"2016-01-{i+start:02d}T06:00").load().sel(lat=slice(-30, 30), lon=slice(-10, 50))

    im = data.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=270, vmax=310, cmap="turbo", add_colorbar=False, rasterized=True)

    ax.set_title(f"{i+start:02d} Jan", fontsize=title_fontsize)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")


cax = fig.add_axes([
    axs[0][-1].get_position().x1+0.01, 
    axs[0][-1].get_position().y0, 
    0.01, 
    axs[0][-1].get_position().height])
cbar = plt.colorbar(im, cax=cax)
cax.tick_params(labelsize=tick_fontsize)
cbar.set_label("ERA5 2mT (K)", fontsize=cbar_title_fontsize, labelpad=10)
cbar.set_ticks([270,290,310])#
shift_cbar_ticklabels(cbar, dy_points=5)



# second row - anomaly -----------------------------------------------------------------

for i, ax in enumerate(axs[1]):
    data = jan_2mt_anom.sel(datetime=f"2016-01-{i+start:02d}").load().sel(lat=slice(-30, 30), lon=slice(-10, 50))

    im = data.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=-10, vmax=10, cmap="RdBu_r", add_colorbar=False, rasterized=True)
    
    ax.set_title("")
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")

cax = fig.add_axes([
    axs[1][-1].get_position().x1+0.01, 
    axs[1][-1].get_position().y0, 
    0.01, 
    axs[1][-1].get_position().height])
cbar = plt.colorbar(im, cax=cax)
cax.tick_params(labelsize=tick_fontsize)
cbar.set_label("ERA5 T2m anom. (K)", fontsize=cbar_title_fontsize)
cbar.set_ticks([-10,0,10])
shift_cbar_ticklabels(cbar, dy_points=5)  # Shift colorbar tick labels down by 10 points

# third row - error --------------------------------------------------------------------


for i, ax in enumerate(axs[2]):
    data = error_data_6am["2m_temperature"].sel(datetime=f"2016-01-{i+start:02d}").load().sel(lat=slice(-30, 30), lon=slice(-10, 50))

    im = data.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=-5, vmax=5, cmap="RdBu_r", add_colorbar=False, rasterized=True)

    ax.set_title("")
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")

cax = fig.add_axes([
    axs[2][-1].get_position().x1+0.01, 
    axs[2][-1].get_position().y0, 
    0.01, 
    axs[2][-1].get_position().height])
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("GC 2mT err. (K)", fontsize=cbar_title_fontsize, labelpad=15)
cax.tick_params(labelsize=tick_fontsize)
cbar.set_ticks([-5,0,5])
shift_cbar_ticklabels(cbar, dy_points=5) 

plt.savefig("africa_error_map_timeseries.pdf", bbox_inches="tight", dpi=300)


## Three timeseries plots on top of each other

In [ ]:
fig, axs = plt.subplots(3, 1, figsize = (10,8))
ylabel_fontsize = 14

axs[0].hlines(y=bust_threshold, xmin=error_series_6am.datetime.min(), xmax=error_series_6am.datetime.max(), color="red", linestyle="--", linewidth=1)
axs[0].plot(ifs_error_series_6am.datetime, ifs_error_series_6am, label="IFS Background - Analysis", color="orange")
axs[0].plot(error_series_6am.datetime, error_series_6am, label="GraphCast - ERA5", color="darkblue")
axs[0].legend(loc="lower right")
axs[0].set_ylabel("2mT error (K)", fontsize=ylabel_fontsize, labelpad=7)


# add in y=0 axis
axs[0].hlines(y=0, xmin=error_series_6am.datetime.min(), xmax=error_series_6am.datetime.max(), color="black", linestyle="-", linewidth=1)

axs[1].plot(era5_series_6am.datetime, era5_series_6am, label="ERA5 Analysis", color="darkblue")
axs[1].plot(gc_series_6am.datetime, gc_series_6am, label="GraphCast 6hr Forecast", color="tab:olive")
axs[1].legend(loc="upper right")
axs[1].set_ylabel("2mT (K)", fontsize=ylabel_fontsize)
axs[1].set_yticks([290,295,300,305])
axs[1].set_ylim(290,305)

axs[2].plot(ifs_analysis_series_6am.datetime, ifs_analysis_series_6am, label="IFS Operational Analysis", color="darkblue")
axs[2].plot(ifs_forecast_series_6am.datetime, ifs_forecast_series_6am, label="IFS Background Forecast", color="tab:olive")
axs[2].legend(loc="upper right")
axs[2].set_ylabel("2mT (K)", fontsize=ylabel_fontsize)
axs[2].legend()
axs[2].set_yticks([290,295,300,305])
axs[2].set_ylim(290,305)

for ax in axs:
    for error_date in bust_dates:
        ax.axvline(error_date.to_numpy(), color="red", linestyle="--", linewidth=1, alpha=0.5)

    ax.set_xlim(np.datetime64("2016-01-01"), np.datetime64("2016-07-01"))
    ax.set_xticks(pd.date_range("2016-02-01", "2016-07-01", freq="2MS"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.set_xticklabels(pd.date_range("2016-02-01", "2016-07-01", freq="2MS").strftime("%b %Y"), fontsize=13)
    ax.tick_params(axis="y", labelsize=13)

# Panel labels (a, b, c)
for i, ax in enumerate(axs):
    ax.text(-0.1, 1.15, f"{chr(97+i)})", transform=ax.transAxes,
            ha="left", va="top", fontsize=13, fontweight="bold")

plt.savefig("forecast_and_error_timeseries.pdf", bbox_inches="tight")

## Distributions comparing performance on forecasts at 12UTC, initialised on bust data

In [ ]:
error_data_12h = all_error_data.where((all_error_data.datetime.dt.hour==12), drop=True)
bust_dates_12h = bust_dates + pd.Timedelta(hours=6) # shift bust dates by 6hrs so they can be used to index the above dataset

In [ ]:

err_bust = error_data_12h.sel(lat=slice(7.5,13.5), lon=slice(35,41))["2m_temperature"].where(error_data_12h.datetime.isin(bust_dates_12h), drop=True)
err_non_bust = error_data_12h.sel(lat=slice(7.5,13.5), lon=slice(35,41))["2m_temperature"].where(~error_data_12h.datetime.isin(bust_dates_12h), drop=True)

avg_err_bust = err_bust.mean(dim="datetime").values.flatten()
avg_err_non_bust = err_non_bust.mean(dim="datetime").values.flatten()

err_bust = err_bust.values.flatten()
err_non_bust = err_non_bust.values.flatten()


fig, axs = plt.subplots(1, 2, figsize=(16,8))
avg_ax = axs[0]
non_avg_ax = axs[1]

avg_bust_counts, avg_bust_bins = np.histogram(avg_err_bust, bins=np.arange(-1, 1.05, 0.05))
avg_bust_counts = avg_bust_counts/avg_bust_counts.sum()
avg_non_bust_counts, avg_non_bust_bins = np.histogram(avg_err_non_bust, bins=np.arange(-1, 1.05, 0.05))
avg_non_bust_counts = avg_non_bust_counts/avg_non_bust_counts.sum()

all_bust_counts, all_bust_bins = np.histogram(err_bust, bins=np.arange(-5, 5.05, 0.05))
all_bust_counts = all_bust_counts/all_bust_counts.sum()
all_non_bust_counts, all_non_bust_bins = np.histogram(err_non_bust, bins=np.arange(-5, 5.05, 0.05))
all_non_bust_counts = all_non_bust_counts/all_non_bust_counts.sum()

avg_ax.stairs(avg_bust_counts, avg_bust_bins, alpha=0.5, label="Error events", color="darkblue", fill=True)
avg_ax.stairs(avg_non_bust_counts, avg_non_bust_bins, alpha=0.5, label="Non-error\nevents", color="orange", fill=True)

non_avg_ax.stairs(all_bust_counts, all_bust_bins, alpha=0.5, color="darkblue", fill=True)
non_avg_ax.stairs(all_non_bust_counts, all_non_bust_bins, alpha=0.5, color="orange", fill=True)


fontsize=30
labelsize=35

plt.subplots_adjust(wspace=0.5)
avg_ax.set_yticks([0,0.05,0.1,0.15])
avg_ax.set_xticks([-1,0,1])
avg_ax.tick_params(axis='both', which='major', labelsize=fontsize)
avg_ax.set_ylabel(r"$p\left(\overline{\epsilon}\right)$", fontsize=labelsize)
avg_ax.set_xlabel(r"Average error, $\overline{\epsilon}$ (K)", fontsize=labelsize)
avg_ax.spines[["right", "top"]].set_visible(False)
avg_ax.text(0.05, 0.95, "a)", transform=avg_ax.transAxes, fontsize=labelsize, fontweight='bold', ha="left", va="top")

non_avg_ax.set_yticks([0,0.01,0.02])
non_avg_ax.set_xticks([-4,0,4])
non_avg_ax.tick_params(axis='both', which='major', labelsize=fontsize)
non_avg_ax.set_ylabel(r"$p\left({\epsilon}\right)$", fontsize=labelsize)
non_avg_ax.set_xlabel(r"Error, $\epsilon$ (K)", fontsize=labelsize)
non_avg_ax.spines[["right", "top"]].set_visible(False)
non_avg_ax.text(0.05, 0.95, "b)", transform=non_avg_ax.transAxes, fontsize=labelsize, fontweight='bold', ha="left", va="top")


avg_ax.legend(loc="upper right", fontsize=25, frameon=True, bbox_to_anchor=(1.3, 1.1), ncols=1)
plt.savefig("summer_figs/gc_12utc_err_hists.pdf", bbox_inches='tight')

print(r"Mean of $p\left(\overline{\epsilon}\right)$, bust days:", avg_err_bust.mean())
print(r"Mean of $p\left(\overline{\epsilon}\right)$, non-bust days:", avg_err_non_bust.mean())

print(r"Mean of $p\left({\epsilon}\right)$, bust days:", err_bust.mean())
print(r"Mean of $p\left({\epsilon}\right)$, non-bust days:", err_non_bust.mean())



## Spread-error plots

In [ ]:
# Addis Ababa and Nairobi airport locations
aa_loc = (9.0, 38.75)
na_loc = (-1.25, 37.0)

spread = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/t2m_spread_2014_16.nc"))

def bin_by_spread(loc, err_data, spread_data, bin_size):

    spread_loc = spread_data.where(spread_data.datetime.dt.hour==6, drop=True).sel(lat=loc[0], lon=loc[1], method="nearest")["t2m"][:-1] # spread data has one extra datetime point at the end, so remove it
    err_loc = err_data.sel(lat=loc[0], lon=loc[1], method="nearest")["2m_temperature"]

    # compute mse and mss
    err_spread_vects = np.array([err_loc.values, spread_loc.values]).T**2
    # sort by spread
    spread_sorted = err_spread_vects[err_spread_vects.copy()[:, 1].argsort()]

    spread_bins = [spread_sorted[i:i+bin_size] for i in range(0, len(spread_sorted), bin_size)]

    spread_binned = np.sqrt(np.array([[np.mean(bin[:, 0]), np.mean(bin[:, 1])] for bin in spread_bins]))

    return spread_binned # format is (rmse, rmss)

aa_binned = bin_by_spread(aa_loc, error_data_6am, spread, 110)# bin size of 110 gives 10 similarly-sized bins for 1095 datapoints
na_binned = bin_by_spread(na_loc, error_data_6am, spread, 110)

In [ ]:

fig, axs = plt.subplots(1, 2, figsize=(12,6))
plt.subplots_adjust(wspace=0.4)

fontsize=20
labelsize=25

axs[0].scatter(aa_binned[:, 1], aa_binned[:, 0], color="darkblue")
axs[0].set_title("Addis Ababa", fontsize=labelsize) #  (9°N, 38.75°E)
axs[0].set_xlim(0,1.75)
axs[0].set_ylim(0,1.75)
axs[0].set_xticks([0,0.5,1,1.5])
axs[0].set_yticks([0,0.5,1,1.5])
axs[0].set_yticklabels(["0", "0.5", "1.0", "1.5"])
axs[0].set_xticklabels(["0", "0.5", "1.0", "1.5"])
axs[0].text(-0.2, 1.05, "a)", transform=axs[0].transAxes, fontsize=labelsize, fontweight='bold', ha="left", va="top")

axs[1].scatter(na_binned[:, 1], na_binned[:, 0], color="darkblue")
axs[1].set_title("Nairobi", fontsize=labelsize) #  (1.25°S, 37°E)
axs[1].set_xlim(0,0.75)
axs[1].set_ylim(0,0.75)
axs[1].set_xticks([0,0.2, 0.4, 0.6])
axs[1].set_yticks([0,0.2, 0.4, 0.6])
axs[1].set_yticklabels(["0", "0.2", "0.4", "0.6"])
axs[1].set_xticklabels(["0", "0.2", "0.4", "0.6"])
axs[1].text(-0.2, 1.05, "b)", transform=axs[1].transAxes, fontsize=labelsize, fontweight='bold', ha="left", va="top")


for ax in axs:
    ax.plot([0,5], [0,5], color="black", linestyle="--")
    ax.set_xlabel("RMS Spread (K)", fontsize=labelsize)
    ax.set_ylabel("RMSE (K)", fontsize=labelsize)
    ax.tick_params(which="both", labelsize=fontsize)

plt.savefig("summer_figs/error_vs_spread_comparison.pdf", bbox_inches="tight")

## Bias map

In [ ]:
africa_bias = error_data_6am.where(~error_data_6am.datetime.isin(bust_dates), drop=True)["2m_temperature"].mean(dim=["datetime"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={"projection": ccrs.PlateCarree()})

africa_bias.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    add_colorbar=True,
    vmin=-1,
    vmax=1,
    rasterized=True
)
ax.coastlines()
ax.set_extent([-10, 50, -30, 30])
ax.add_feature(cfeature.BORDERS, linestyle=':')

ax.add_patch(plt.Rectangle((35, 7.5), 6, 6, fill=False, color='black', linewidth=2))

cbar = ax.collections[0].colorbar
cax = cbar.ax

cbar.set_ticks([-1,-0.5,0,0.5,1])
cax.tick_params(labelsize=15)
cbar.set_label("2mT Bias (K)", fontsize=15)

plt.savefig("summer_figs/6am_bias_africa_2014_16.pdf", bbox_inches="tight", dpi=300)

# Statistics/other info

## EOFs at all timesteps

In [ ]:
model_00 = pickle.load(open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/experiments/ea_errors_00_14_16/model.pkl","rb"))
model_06 = pickle.load(open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/experiments/ea_errors_06_14_16/model.pkl","rb"))
model_12 = pickle.load(open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/experiments/ea_errors_12_14_16/model.pkl","rb"))
model_18 = pickle.load(open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/experiments/ea_errors_18_14_16/model.pkl","rb"))
eofs_2mt_00 = model_00.get_components(10)["2m_temperature"].reshape(10,721,1440)
eofs_2mt_06 = model_06.get_components(10)["2m_temperature"].reshape(10,721,1440)
eofs_2mt_12 = model_12.get_components(10)["2m_temperature"].reshape(10,721,1440)
eofs_2mt_18 = model_18.get_components(10)["2m_temperature"].reshape(10,721,1440)

In [ ]:
for j, eofs in enumerate([eofs_2mt_00, eofs_2mt_06, eofs_2mt_12, eofs_2mt_18]):
    fig, axs = plt.subplots(2, 5, figsize=(14, 7), subplot_kw={'projection': ccrs.PlateCarree()},
    gridspec_kw={"wspace": 0.15, "hspace": -0.08})

    extent=[0, 360, -90, 90]

    for i, ax in enumerate(axs.flatten()):
        im = ax.imshow(
            eofs[i],
            cmap="RdBu_r",
            origin="lower",
            extent=extent,
            transform=ccrs.PlateCarree(),
            vmin=-0.02,
            vmax=0.02,
            rasterized=True
        )
        ax.coastlines()
        ax.set_extent([-10,50,-30,30])
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"EOF {i+1} {j*6}UTC")
    plt.show()


## Bust day statistics over different years and times

In [ ]:
errors_1980 = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_error_gc_runs/graphcast_full/graphcast_predictions/errors_{date.strftime('%Y%m%d-%H')}_n0.nc" for date in pd.date_range("1980-01-01", "1980-12-31-18:00", freq="6h")],
    combine="nested",
    concat_dim="datetime",
    preprocess=preprocess
)
# pred_1980 = xr.open_mfdataset(
#     [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_error_gc_runs/graphcast_full/graphcast_predictions/pred_{date.strftime('%Y%m%d')}-06_n0.nc" for date in pd.date_range("1980-01-01", "1980-12-31", freq="D")],
#     combine="nested",
#     concat_dim="time",
#     preprocess=preprocess
# ).rename({"time": "datetime"}).set_index({"datetime": "datetime"})

errors_1990 = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_error_gc_runs/graphcast_full/graphcast_predictions/errors_{date.strftime('%Y%m%d-%H')}_n0.nc" for date in pd.date_range("1990-01-01", "1990-12-31-18:00", freq="6h")],
    combine="nested",
    concat_dim="datetime",
    preprocess=preprocess
)
# pred_1990 = xr.open_mfdataset(
#     [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_error_gc_runs/graphcast_full/graphcast_predictions/pred_{date.strftime('%Y%m%d')}-06_n0.nc" for date in pd.date_range("1990-01-01", "1990-12-31", freq="D")],
#     combine="nested",
#     concat_dim="time",
#     preprocess=preprocess
# ).rename({"time": "datetime"}).set_index({"datetime": "datetime"})


In [ ]:
ds1 = xr.open_dataset(os.path.join("/network/group/aopp/predict/HMC005_ANTONIO_EERIE/predictions/graphcast_full/jack_graphcast_errors/errors_20001027-18_n0.nc"))

In [ ]:
ds2 = xr.open_dataset(os.path.join("/network/group/aopp/predict/HMC005_ANTONIO_EERIE/predictions/graphcast_full/jack_graphcast_errors/errors_20001028-00_n0.nc"))

In [ ]:

errors_2000 = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC005_ANTONIO_EERIE/predictions/graphcast_full/jack_graphcast_errors/errors_{date.strftime('%Y%m%d-%H')}_n0.nc" for date in pd.date_range("2000-01-01", "2000-12-31-18:00", freq="6h")],
    combine="nested",
    concat_dim="datetime",
    preprocess=preprocess
)


In [ ]:

errors_2010 = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC005_ANTONIO_EERIE/predictions/graphcast_full/jack_graphcast_errors/errors_{date.strftime('%Y%m%d-%H')}_n0.nc" for date in pd.date_range("2010-01-01", "2010-12-31-18:00", freq="6h")],
    combine="nested",
    concat_dim="datetime",
    preprocess=preprocess
)


In [ ]:

# -1 averages at all times of day.
# mfdataset uses lazy loading, so call .load() once highest level of selection is performed
error_timeseries_1980 = average_area("1980-01-01", "1980-12-31", -1, errors_1980["2m_temperature"]).load()
error_timeseries_1990 = average_area("1990-01-01", "1990-12-31", -1, errors_1990["2m_temperature"]).load()
error_timeseries_2000 = average_area("2000-01-01", "2000-12-31", -1, errors_2000["2m_temperature"]).load()
error_timeseries_2010 = average_area("2010-01-01", "2010-12-31", -1, errors_2010["2m_temperature"]).load()
# error_timeseries_2014_16 = average_area("2014-01-01", "2016-12-31", -1, all_error_data["2m_temperature"])

In [ ]:
bust_timestamps_1980 = error_timeseries_1980.datetime[error_timeseries_1980 < bust_threshold]
bust_timestamps_1990 = error_timeseries_1990.datetime[error_timeseries_1990 < bust_threshold]
bust_timestamps_2000 = error_timeseries_2000.datetime[error_timeseries_2000 < bust_threshold]
bust_timestamps_2010 = error_timeseries_2010.datetime[error_timeseries_2010 < bust_threshold]
# bust_timestamps_2014_16 = error_timeseries_2014_16.datetime[error_timeseries_2014_16 < bust_threshold]

for hr in [0,6,12,18]:
    print(f"Number of busts in 1980 at {hr}UTC: {bust_timestamps_1980[bust_timestamps_1980.dt.hour == hr].size}, {100*error_timeseries_1980[error_timeseries_1980.dt.hour == hr].size:0.1f}%")
    print(f"Number of busts in 1990 at {hr}UTC: {bust_timestamps_1990[bust_timestamps_1990.dt.hour == hr].size}, {100*error_timeseries_1990[error_timeseries_1990.dt.hour == hr].size:0.1f}%")
    print(f"Number of busts in 2000 at {hr}UTC: {bust_timestamps_2000[bust_timestamps_2000.dt.hour == hr].size}, {100*error_timeseries_2000[error_timeseries_2000.dt.hour == hr].size:0.1f}%")
    print(f"Number of busts in 2010 at {hr}UTC: {bust_timestamps_2010[bust_timestamps_2010.dt.hour == hr].size}, {100*error_timeseries_2010[error_timeseries_2010.dt.hour == hr].size:0.1f}%")
    # print(f"Number of busts in 2014-16 at {hr}UTC: {bust_timestamps_2014_16[bust_timestamps_2014_16.dt.hour == hr].size}")


In [ ]:
all_error_data = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/gc_data/2mt_errors_2014_16.nc")
error_data_6am = all_error_data.where(all_error_data.datetime.dt.hour == 6, drop=True)
all_pred_data = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/gc_data/2mt_pred_2014_16.nc")
pred_data_6am = all_pred_data.where(all_pred_data.datetime.dt.hour == 6, drop=True)
era5_data = all_pred_data - all_error_data
era5_data_6am = era5_data.where(era5_data.datetime.dt.hour == 6, drop=True)

error_series_6am = average_area("2014-01-01", "2016-12-31", 6, error_data_6am["2m_temperature"])

In [ ]:
error_timeseries_2000_6am = average_area("2000-01-01", "2000-12-31", 6, errors_2000["2m_temperature"]).load()

In [ ]:
bust_threshold=-1.5

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (10,5))
ylabel_fontsize = 14

axs.hlines(y=bust_threshold, xmin=error_timeseries_2000_6am.datetime.min(), xmax=error_timeseries_2000_6am.datetime.max(), color="red", linestyle="--", linewidth=1)
# axs[0].plot(ifs_error_series_6am.datetime, ifs_error_series_6am, label="IFS Background - Analysis", color="orange")
axs.plot(error_timeseries_2000_6am.datetime, error_timeseries_2000_6am, label="GraphCast - ERA5", color="darkblue")
axs.legend(loc="lower right")
axs.set_ylabel("2mT error (K)", fontsize=ylabel_fontsize, labelpad=7)


# # add in y=0 axis
# axs[0].hlines(y=0, xmin=error_series_6am.datetime.min(), xmax=error_series_6am.datetime.max(), color="black", linestyle="-", linewidth=1)

# axs[1].plot(era5_series_6am.datetime, era5_series_6am, label="ERA5 Analysis", color="darkblue")
# axs[1].plot(gc_series_6am.datetime, gc_series_6am, label="GraphCast 6hr Forecast", color="tab:olive")
# axs[1].legend(loc="upper right")
# axs[1].set_ylabel("2mT (K)", fontsize=ylabel_fontsize)
# axs[1].set_yticks([290,295,300,305])
# axs[1].set_ylim(290,305)

# for ax in axs:
#     for error_date in bust_dates:
#         ax.axvline(error_date.to_numpy(), color="red", linestyle="--", linewidth=1, alpha=0.5)

#     ax.set_xlim(np.datetime64("2016-01-01"), np.datetime64("2016-07-01"))
#     ax.set_xticks(pd.date_range("2016-02-01", "2016-07-01", freq="2MS"))
#     ax.xaxis.set_minor_locator(mdates.MonthLocator())
#     ax.set_xticklabels(pd.date_range("2016-02-01", "2016-07-01", freq="2MS").strftime("%b %Y"), fontsize=13)
#     ax.tick_params(axis="y", labelsize=13)

# # Panel labels (a, b, c)
# for i, ax in enumerate(axs):
#     ax.text(-0.1, 1.15, f"{chr(97+i)})", transform=ax.transAxes,
#             ha="left", va="top", fontsize=13, fontweight="bold")

# plt.savefig("summer_figs/forecast_and_error_timeseries.pdf", bbox_inches="tight")

In [ ]:
pd.date_range("2010-01-01", "2010-12-31-18:00", freq="6h")

## Significance of bias over Ethiopia seen on non-bust days
Answering the question - does GC introduce a statistically significant bias on non-bust days, in order to "hedge" the bust days?

In [ ]:
land_mask = np.load("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/land_mask.npy")
land_mask = xr.DataArray(land_mask, dims=["lat", "lon"], coords={"lat": error_data_6am.coords["lat"], "lon": error_data_6am.coords["lon"]})


# randomly generate n latitude and longitude points
n_points = 1000000
latitudes = np.random.uniform(low=-90, high=84, size=n_points)
longitudes = np.random.uniform(low=0, high=354, size=n_points)

# round each point down to the nearest 0.25
latitudes = np.floor(latitudes * 4) / 4
longitudes = np.floor(longitudes * 4) / 4

# for each point, take a 6x6 square and check if it contains any sea
# discard any squares that do contain sea.

valid_points = []

for (lat, lon) in zip(latitudes, longitudes):
    # immediately rule out points in the sea
    if land_mask.sel(lat=lat, lon=lon).values != 0:
        subset = land_mask.sel(lat=slice(lat, lat+6), lon=slice(lon, lon+6)).values.flatten()
        if not(subset[subset == 0].size > 0):
            valid_points.append((lat, lon))

len(valid_points)

In [ ]:
# for each valid point, we want to compute the time-averaged mean error over the region
mean_errors = np.zeros(len(valid_points))

averaging_data = error_data_6am.where(~error_data_6am.datetime.isin(bust_dates), drop=True)

for i, (lat, lon) in enumerate(valid_points):
    val = averaging_data["2m_temperature"].sel(lat=slice(lat, lat+6), lon=slice(lon, lon+6)).mean(dim=['lat', 'lon', "datetime"]).values
    mean_errors[i] = val


In [ ]:
print(f"Distribution mean: {mean_errors.mean():.4f}, std: {mean_errors.std():.4f}")

# work out if the average over the 6degx6deg Ethiopia square is significantly different
mean_err_eth = averaging_data["2m_temperature"].sel(lat=slice(7.5, 13.5), lon=slice(35, 41)).mean(dim=['lat', 'lon', "datetime"]).values

percentile = sum(mean_errors < mean_err_eth) / len(mean_errors)

print(f"Ethiopia bias: {mean_err_eth:.4f}")
print(f"z_score of Ethiopia bias: {(mean_err_eth - mean_errors.mean()) / mean_errors.std():.2f}")
print(f"Percentile of Ethiopia bias: {percentile * 100:.2f}%")

opposite_percentile = sum(mean_errors < -mean_err_eth) / len(mean_errors)
print(f"Percentile of -(Ethiopia bias): {opposite_percentile * 100:.2f}%")
print(f"Percentile level of |bias|: {(1-percentile + opposite_percentile)*100:.2f}")

In [ ]:
hist, bin_edges = np.histogram(mean_errors, bins=30)

ticklabelsize = 15
axistitlesize = 15
fig, ax = plt.subplots(figsize=(6, 4.5))

hist = hist / hist.sum()
ax.stairs(hist, bin_edges, fill=True, color="darkblue")
ax.set_xlim(-0.3, 0.3)

ax.set_xlabel("Bias (K)", fontsize=axistitlesize)
ax.set_ylabel(r"$p\left(\text{Bias}\right)$", fontsize=axistitlesize)
ax.tick_params(labelsize=ticklabelsize)
# add arrow
ax.annotate(
    'Ethiopia', 
    xy=(mean_err_eth, 0.005), 
    xytext=(mean_err_eth-0.055, 0.04),
    color="red",
    fontsize=ticklabelsize,
    arrowprops=dict(arrowstyle='->', color='red', linewidth=2))

plt.savefig("summer_figs/bias_dist.pdf", bbox_inches='tight')

# Supplementary materials

## 2mT errors at 4,5,6,7,8UTC

In [ ]:
# load era5 data for the area at other timesteps

tickfontsize=13

era5_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016_more_times.nc"))
times = average_area("2016-01-01", "2016-12-31", 6, era5_data["t2m"]).datetime

fig, axs = plt.subplots(5, 1, figsize=(12, 13), sharex=True)

for i, ax in enumerate(axs):
    axs[i].plot(times, average_area("2016-01-01", "2016-12-31", i+4, era5_data["t2m"]))
    axs[i].set_title(f"0{i+4}:00UTC", fontsize=tickfontsize)
    axs[i].set_ylabel("Temp. (K)", fontsize=tickfontsize)
    axs[i].tick_params(which="both", labelsize=tickfontsize)

    for bust_date in bust_dates:
        axs[i].axvline(bust_date.to_numpy(), color='red', linestyle='--', alpha=0.5)

    ax.set_ylim(285, 305)
    axs[i].set_yticks([290, 295, 300])

ax.set_xlim(np.datetime64("2016-01-01"), np.datetime64("2016-12-31"))
ax.set_xticks(pd.date_range("2016-02-01", "2016-12-01", freq="2MS"))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_xticklabels(pd.date_range("2016-02-01", "2016-12-01", freq="2MS").strftime("%b %Y"))

plt.savefig("summer_figs/sup_mat_2mt_diff_times.pdf", bbox_inches="tight")


## Plots of other temperature variables

In [ ]:
temp_levels_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_t_levels_2016.nc"))
skint_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_skint_2016.nc"))

temp_time_series = average_area("2016-01-01", "2016-12-31", 6, temp_levels_data["t"])
skint_time_series = average_area("2016-01-01", "2016-12-31", 6, skint_data["skt"])

t2m_series = era5_series_6am.where(era5_series_6am.datetime.dt.year==2016, drop=True)

In [ ]:
fig, axs = plt.subplots(len(temp_levels_data.pressure_level)+2, 1, figsize=(13, 15), sharex=True)
plt.subplots_adjust(hspace=0.4)

axs[0].plot(t2m_series.datetime, t2m_series, label="ERA5 2m Temperature")
axs[0].set_title("2m Temperature")

axs[1].plot(skint_time_series.datetime, skint_time_series, label="ERA5 SkinTemperature")
axs[1].set_title("Skin Temperature")

for i, ax in enumerate(axs):
    for error_time in bust_dates:
        axs[i].axvline(error_time.to_numpy(), color='red', linestyle='--', alpha=0.5)
        ax.set_xlim(np.datetime64("2016-01-01"), np.datetime64("2016-12-31"))
    axs[i].set_ylabel("Temp. (K)")

for i, level in enumerate(temp_levels_data.pressure_level):
    axs[i+2].plot(temp_time_series.datetime, temp_time_series.isel(pressure_level=i), label=f"ERA5 {level}hPa Temperature")
    axs[i+2].set_title(f"{level.data:.0f}hPa Temperature")
    

ax.set_xticks(pd.date_range("2016-02-01", "2016-12-01", freq="2MS"))
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.set_xticklabels(pd.date_range("2016-02-01", "2016-12-01", freq="2MS").strftime("%b %Y"), fontsize=13)

axs[4].set_yticks([285, 290])
axs[5].set_yticks([282, 284, 286])
axs[-1].set_yticks([265, 270])

plt.savefig("summer_figs/sup_mat_t_plots.pdf", bbox_inches="tight")


## Stamp plots of Africa on error days in other years

In [ ]:
bust_dates_1980 = bust_timestamps_1980[bust_timestamps_1980.dt.hour == 6]
len(bust_dates_1980)

In [ ]:
fig, axs = plt.subplots(
    3, 
    6, 
    figsize=(20, 8), 
    subplot_kw={"projection": ccrs.PlateCarree()},
    gridspec_kw={"wspace": -0.70, "hspace": 0.2})


title_fontsize = 15
tick_fontsize = 15
cbar_title_fontsize = 15



for i, ax in enumerate(axs.flatten()):
    data = errors_1980["2m_temperature"].sel(datetime=bust_dates_1980[i]).sel(lat=slice(-30, 30), lon=slice(-10, 50)).load()

    im = data.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=-5, vmax=5, cmap="RdBu_r", add_colorbar=False, rasterized=True)

    ax.set_title(f"{bust_dates_1980[i].dt.strftime('%Y-%m-%d').values}", fontsize=title_fontsize)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")


cax = fig.add_axes([
    axs[-1][-1].get_position().x1+0.01, 
    axs[-1][-1].get_position().y0+0.05, 
    0.01, 
    axs[-1][-1].get_position().height*3])
cbar = plt.colorbar(im, cax=cax)
cax.tick_params(labelsize=tick_fontsize)
cbar.set_label("2mT Error (K)", fontsize=cbar_title_fontsize, labelpad=10)
shift_cbar_ticklabels(cbar, dy_points=5)

plt.savefig("summer_figs/sup_mat_err_maps_1980.pdf", bbox_inches="tight", dpi=300)


In [ ]:
bust_dates_1990 = bust_timestamps_1990[bust_timestamps_1990.dt.hour == 6]
len(bust_dates_1990)

In [ ]:
fig, axs = plt.subplots(
    2, 
    6, 
    figsize=(20, 8*2/3), 
    subplot_kw={"projection": ccrs.PlateCarree()},
    gridspec_kw={"wspace": -0.70, "hspace": 0.2})


title_fontsize = 15
tick_fontsize = 15
cbar_title_fontsize = 15

for i, ax in enumerate(axs.flatten()):
    data = errors_1990["2m_temperature"].sel(datetime=bust_dates_1990[i]).sel(lat=slice(-30, 30), lon=slice(-10, 50)).load()

    im = data.plot(ax=ax, transform=ccrs.PlateCarree(), vmin=-5, vmax=5, cmap="RdBu_r", add_colorbar=False, rasterized=True)

    ax.set_title(f"{bust_dates_1990[i].dt.strftime('%Y-%m-%d').values}", fontsize=title_fontsize)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")


cax = fig.add_axes([
    axs[-1][-1].get_position().x1+0.01, 
    axs[-1][-1].get_position().y0+0.05, 
    0.01, 
    axs[-1][-1].get_position().height*2])
cbar = plt.colorbar(im, cax=cax)
cax.tick_params(labelsize=tick_fontsize)
cbar.set_label("2mT Error (K)", fontsize=cbar_title_fontsize, labelpad=10)
shift_cbar_ticklabels(cbar, dy_points=5)

plt.savefig("summer_figs/sup_mat_err_maps_1990.pdf", bbox_inches="tight", dpi=300)


## EOF plots

In [ ]:
model_06 = pickle.load(open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/experiments/ea_errors_06_14_16/model.pkl","rb"))
eofs_2mt = model_06.get_components(10)["2m_temperature"].reshape(10,721,1440)

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(14, 7), subplot_kw={'projection': ccrs.PlateCarree()},
gridspec_kw={"wspace": 0.15, "hspace": -0.08})

extent=[0, 360, -90, 90]

for i, ax in enumerate(axs.flatten()):
    im = ax.imshow(
        eofs_2mt[i],
        cmap="RdBu_r",
        origin="lower",
        extent=extent,
        transform=ccrs.PlateCarree(),
        vmin=-0.02,
        vmax=0.02,
        rasterized=True
    )
    ax.coastlines()
    ax.set_extent([-10,50,-30,30])
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.set_title(f"EOF {i+1}", fontsize=15)

plt.savefig("summer_figs/sup_mat_eofs_2mt.pdf", bbox_inches="tight", dpi=300)
